# Data Description
- Background: Unit 1.7 補充資料驗證：臺北事故斑點圖。提供機關：臺北市警察局交通大隊；官方來源：https://data.taipei/dataset/detail?id=0554bac7-cbc2-4ef3-a55e-0aad3dd4ee1d。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/113年臺北市道路交通事故斑點圖 .csv`；1,580,764 bytes；SHA-256 `63dbff7974735ab16ed000f344b72d5a7b9ffc6106e1dfb8030b666e99ddd45c`。
  - `data/raw/114年臺北市道路交通事故斑點圖.csv`；1,606,943 bytes；SHA-256 `30851896b0df6000c5bca3032ecf5ad05baae04fa6973bd563ee77341cb958eb`。
- Unit of observation: 事故位置公開紀錄，無官方事件ID；與第一當事人候選事件對照。
- Sample period: 事故發生期間2024／2025；下載日未知；發布／更新：113資源2025-04-10；114資源2026-04-21。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 114原始CP950含非法byte；可逆surrogateescape只作診斷，受損地址不得確認精確配對。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='accident_points'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,accident_points,data/raw/113年臺北市道路交通事故斑點圖 .csv,1580764,63dbff7974735ab16ed000f344b72d5a7b9ffc6106e1dfb8030b666e99ddd45c,present
1,accident_points,data/raw/114年臺北市道路交通事故斑點圖.csv,1606943,30851896b0df6000c5bca3032ecf5ad05baae04fa6973bd563ee77341cb958eb,present


**讀取範圍與CRS／編碼**

,path,rows,encoding,strict,byte_roundtrip,delimiter,invalid_byte_count,invalid_bytes
0,data/raw/113年臺北市道路交通事故斑點圖 .csv,22368,cp950,True,True,"','",0,[]
1,data/raw/114年臺北市道路交通事故斑點圖.csv,22762,cp950,False,True,"','",1,"[{'byte_offset': 42947, 'physical_line': 620, 'hex': '5fb8f4a56080b944292c31'}]"


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/113年臺北市道路交通事故斑點圖 .csv**

,發生時間,處理別,肇事地點,座標-X,座標-Y,_source_row,_source_path,_decoding_valid
0,2024/1/1 07:12,2,大同區延平北路2段144巷與延平北路2段口,121.511493,25.0576205,1,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
1,2024/1/1 15:31,2,大同區環河北路1段與南京西路口,121.5075289,25.0536646,2,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
2,2024/1/1 17:28,2,大同區民生西路與承德路2段口口,121.5183534,25.0574563,3,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
3,2024/1/2 12:30,2,大同區重慶北路3段257巷與重慶北路3段257巷口23號旁通道,121.5148424,25.0715984,4,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
4,2024/1/2 17:24,2,大同區鄭州路與西寧北路口,121.508197,25.0505848,5,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
5,2024/1/2 19:00,2,大同區南京西路與太原路口,121.5161045,25.0535851,6,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
6,2024/1/2 19:44,2,大同區南京西路與南京西路302巷口,121.5140971,25.0538164,7,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
7,2024/1/3 06:27,2,大同區延平北路3段與民族西路口,121.5109737,25.0686873,8,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
8,2024/1/3 07:30,2,大同區承德路3段與民權西路口,121.5185036,25.0630016,9,data/raw/113年臺北市道路交通事故斑點圖 .csv,True
9,2024/1/3 13:18,2,大同區民權西路102巷與民權西路102巷33弄口,121.5176956,25.0615791,10,data/raw/113年臺北市道路交通事故斑點圖 .csv,True


**data/raw/114年臺北市道路交通事故斑點圖.csv**

,發生時間,處理別,肇事地點,座標-X,座標-Y,_source_row,_source_path,_decoding_valid
0,2025/1/1 00:18,2,中山區市民大道2段47號,121.5290038,25.04660303,1,data/raw/114年臺北市道路交通事故斑點圖.csv,True
1,2025/1/1 00:31,2,文山區新光路1段109號旁,121.5736892,24.9912752,2,data/raw/114年臺北市道路交通事故斑點圖.csv,True
2,2025/1/1 00:35,2,中山區北安路與大直街口,121.5471963,25.07984181,3,data/raw/114年臺北市道路交通事故斑點圖.csv,True
3,2025/1/1 00:41,2,中山區民族東路2號,121.5233047,25.0683953,4,data/raw/114年臺北市道路交通事故斑點圖.csv,True
4,2025/1/1 00:57,2,南港區忠孝東路6段與玉成街口,121.5832269,25.0458717,5,data/raw/114年臺北市道路交通事故斑點圖.csv,True
5,2025/1/1 01:34,2,信義區吳興街269巷18弄與吳興街281巷47弄口,121.5652003,25.0269334,6,data/raw/114年臺北市道路交通事故斑點圖.csv,True
6,2025/1/1 02:15,2,信義區基隆路2段107號之2,121.5575723,25.029541,7,data/raw/114年臺北市道路交通事故斑點圖.csv,True
7,2025/1/1 06:26,2,文山區指南路3段(岐山幹42),121.5972232,24.9724278,8,data/raw/114年臺北市道路交通事故斑點圖.csv,True
8,2025/1/1 06:29,2,文山區辛亥路4段與辛亥路4段101巷口,121.5583969,25.0064466,9,data/raw/114年臺北市道路交通事故斑點圖.csv,True
9,2025/1/1 07:11,2,松山區寶清街93號,121.5679891,25.0524534,10,data/raw/114年臺北市道路交通事故斑點圖.csv,True


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,臺北事故斑點圖
provider,臺北市警察局交通大隊
url,https://data.taipei/dataset/detail?id=0554bac7-cbc2-4ef3-a55e-0aad3dd4ee1d
publication,113資源2025-04-10；114資源2026-04-21
period,事故發生期間2024／2025；下載日未知
limitation,114原始CP950含非法byte；可逆surrogateescape只作診斷，受損地址不得確認精確配對
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,accident_points,data/raw/113年臺北市道路交通事故斑點圖 .csv,1580764,63dbff7974735ab16ed000f344b72d5a7b9ffc6106e1dfb8030b666e99ddd45c,present


,family,path,bytes,sha256,status
0,accident_points,data/raw/114年臺北市道路交通事故斑點圖.csv,1606943,30851896b0df6000c5bca3032ecf5ad05baae04fa6973bd563ee77341cb958eb,present


**實際讀取範圍／編碼／CRS／全國或地區計數**

,path,rows,encoding,strict,byte_roundtrip,delimiter,invalid_byte_count,invalid_bytes
0,data/raw/113年臺北市道路交通事故斑點圖 .csv,22368,cp950,True,True,"','",0,[]
1,data/raw/114年臺北市道路交通事故斑點圖.csv,22762,cp950,False,True,"','",1,"[{'byte_offset': 42947, 'physical_line': 620, 'hex': '5fb8f4a56080b944292c31'}]"


### data/raw/113年臺北市道路交通事故斑點圖 .csv

... (21 outputs left out)

,field,missing_or_unparseable,zero
0,座標-X,2,0
1,座標-Y,2,0


原始屬性完整列重複（不含診斷追溯欄） 0


**類別前12項：處理別**

,處理別,N,ratio_of_loaded_rows
0,2,22705,0.997496
1,1,57,0.002504


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
